In [3]:
import re
import string
import json
import warnings
 
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from wordcloud import WordCloud, STOPWORDS as WC_STOPWORDS
 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
 
import nltk
from nltk.corpus import stopwords
 
warnings.filterwarnings("ignore")
 
OUT = "outputs"


1. DATA LOADING


In [6]:
print("Step 1: Loading data...")
df = pd.read_csv("amazonreviews (2).csv", sep="\t")
print(f"  Raw shape: {df.shape}")
print(df['label'].value_counts())



Step 1: Loading data...
  Raw shape: (10000, 2)
label
neg    5097
pos    4903
Name: count, dtype: int64


2. DATA CLEANING

In [5]:
missing_before = df['review'].isna().sum()
df = df.dropna(subset=['review', 'label']).reset_index(drop=True)
print(f"  Dropped {missing_before} rows with missing review/label")


  Dropped 0 rows with missing review/label


In [7]:
dupes_before = df.duplicated(subset=['review']).sum()
df = df.drop_duplicates(subset=['review']).reset_index(drop=True)
print(f"  Dropped {dupes_before} duplicate reviews")


  Dropped 0 duplicate reviews


In [8]:
df['label'] = df['label'].str.strip().str.lower()
df = df[df['label'].isin(['pos', 'neg'])].reset_index(drop=True)
df['target'] = (df['label'] == 'pos').astype(int)


In [9]:
stop_words = set(stopwords.words('english'))
# Keep negation words - they matter a lot for sentiment (e.g. "not good")
negations = {"no", "not", "nor", "n't", "never", "none"}
stop_words = stop_words - negations


In [10]:

def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"<.*?>", " ", text)                 # strip HTML tags
    text = re.sub(r"http\S+|www\.\S+", " ", text)       # strip URLs
    text = re.sub(r"[^a-z\s]", " ", text)               # keep letters only
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
    return " ".join(tokens)
 
df['clean_review'] = df['review'].apply(clean_text)


In [13]:
empty_after_clean = (df['clean_review'].str.strip() == "").sum()
df = df[df['clean_review'].str.strip() != ""].reset_index(drop=True)
print(f"  Dropped {empty_after_clean} rows that were empty after cleaning")
print(f"  Final cleaned shape: {df.shape}")
 
df.to_csv(f"{OUT}/cleaned_reviews_sample.csv", index=False, columns=['label', 'review', 'clean_review'])


  Dropped 0 rows that were empty after cleaning
  Final cleaned shape: (10000, 4)


3. EXPLORATORY DATA ANALYSIS


In [14]:
print("\nStep 3: Exploratory analysis...")
 
# 3a. Sentiment distribution
plt.figure(figsize=(5, 4))
counts = df['label'].value_counts()
plt.bar(counts.index, counts.values, color=['#4C9F70', '#D9534F'])
plt.title("Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Number of Reviews")
for i, v in enumerate(counts.values):
    plt.text(i, v + 20, str(v), ha='center')
plt.tight_layout()
plt.savefig(f"{OUT}/sentiment_distribution.png", dpi=150)
plt.close()



Step 3: Exploratory analysis...


In [15]:
df['review_len'] = df['review'].str.split().apply(len)
plt.figure(figsize=(6, 4))
plt.hist(df[df['label'] == 'pos']['review_len'], bins=40, alpha=0.6, label='Positive', color='#4C9F70')
plt.hist(df[df['label'] == 'neg']['review_len'], bins=40, alpha=0.6, label='Negative', color='#D9534F')
plt.xlabel("Review length (words)")
plt.ylabel("Frequency")
plt.title("Review Length by Sentiment")
plt.legend()
plt.tight_layout()
plt.savefig(f"{OUT}/review_length_distribution.png", dpi=150)
plt.close()


In [16]:
for label, fname, cmap in [('pos', 'wordcloud_positive.png', 'Greens'),
                            ('neg', 'wordcloud_negative.png', 'Reds')]:
    text = " ".join(df[df['label'] == label]['clean_review'].tolist())
    wc = WordCloud(width=900, height=500, background_color='white',
                    colormap=cmap, max_words=150,
                    stopwords=WC_STOPWORDS).generate(text)
    plt.figure(figsize=(9, 5))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title(f"Most Frequent Words - {'Positive' if label=='pos' else 'Negative'} Reviews")
    plt.tight_layout()
    plt.savefig(f"{OUT}/{fname}", dpi=150)
    plt.close()


In [17]:
def top_words(series, n=20):
    from collections import Counter
    words = " ".join(series).split()
    return Counter(words).most_common(n)
 
top_pos = top_words(df[df['label'] == 'pos']['clean_review'])
top_neg = top_words(df[df['label'] == 'neg']['clean_review'])


In [19]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].barh([w for w, c in top_pos[::-1]], [c for w, c in top_pos[::-1]], color='#4C9F70')
axes[0].set_title("Top 20 Words - Positive Reviews")
axes[1].barh([w for w, c in top_neg[::-1]], [c for w, c in top_neg[::-1]], color='#D9534F')
axes[1].set_title("Top 20 Words - Negative Reviews")
plt.tight_layout()
plt.savefig(f"{OUT}/top_words_by_sentiment.png", dpi=150)
plt.close()
 
print(" sentiment_distribution.png, review_length_distribution.png,")
print("         wordcloud_positive.png, wordcloud_negative.png, top_words_by_sentiment.png")
 

 sentiment_distribution.png, review_length_distribution.png,
         wordcloud_positive.png, wordcloud_negative.png, top_words_by_sentiment.png


4. FEATURE EXTRACTION + MODEL DEVELOPMENT


In [20]:
X = df['clean_review']
y = df['target']
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
 
# TF-IDF with unigrams + bigrams captures short sentiment phrases ("not good")
tfidf = TfidfVectorizer(max_features=15000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)
 


In [21]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, C=5),
    "Linear SVM": LinearSVC(C=1),
}
 
results = {}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
 


In [22]:
results = {}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
 
for name, model in models.items():
    print(f"\n  Training: {name}")
    # 5-fold cross-validation on the training set
    cv_scores = cross_val_score(model, X_train_tfidf, y_train, cv=skf, scoring='f1')
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
 
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
 
    results[name] = {
        "cv_f1_mean": float(cv_scores.mean()),
        "cv_f1_std": float(cv_scores.std()),
        "test_accuracy": float(acc),
        "test_f1": float(f1),
        "test_precision": float(prec),
        "test_recall": float(rec),
    }



  Training: Logistic Regression

  Training: Linear SVM


In [23]:
    print(f"    5-fold CV F1: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    print(f"    Test Accuracy: {acc:.4f} | F1: {f1:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f}")


    5-fold CV F1: 0.8713 (+/- 0.0042)
    Test Accuracy: 0.8595 | F1: 0.8566 | Precision: 0.8579 | Recall: 0.8552


In [25]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['neg', 'pos'])
fig, ax = plt.subplots(figsize=(4.5, 4))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f"Confusion Matrix - {name}")
plt.tight_layout()
safe_name = name.lower().replace(" ", "_")
plt.savefig(f"{OUT}/confusion_matrix_{safe_name}.png", dpi=150)
plt.close()



5. MOST INFLUENTIAL WORDS (Logistic Regression coefficients)


In [26]:
print("\nStep 5: Extracting most influential words (Logistic Regression)...")
log_reg = models["Logistic Regression"]
feature_names = np.array(tfidf.get_feature_names_out())
coefs = log_reg.coef_[0]
 
top_pos_idx = np.argsort(coefs)[-15:][::-1]
top_neg_idx = np.argsort(coefs)[:15]
 
top_pos_features = list(zip(feature_names[top_pos_idx], coefs[top_pos_idx]))
top_neg_features = list(zip(feature_names[top_neg_idx], coefs[top_neg_idx]))


Step 5: Extracting most influential words (Logistic Regression)...


In [27]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].barh([w for w, c in top_pos_features[::-1]], [c for w, c in top_pos_features[::-1]], color='#4C9F70')
axes[0].set_title("Strongest Positive Predictors")
axes[1].barh([w for w, c in top_neg_features[::-1]], [c for w, c in top_neg_features[::-1]], color='#D9534F')
axes[1].set_title("Strongest Negative Predictors")
plt.tight_layout()
plt.savefig(f"{OUT}/influential_words_logreg.png", dpi=150)
plt.close()
 

6. SAVE METRICS + SUMMARY JSON FOR THE REPORT


In [30]:
summary = {
    "dataset": {
        "raw_rows": int(pd.read_csv("amazonreviews (2).csv", sep="\t").shape[0]),
        "final_rows_after_cleaning": int(df.shape[0]),
        "duplicates_removed": int(dupes_before),
        "missing_removed": int(missing_before),
        "empty_after_clean_removed": int(empty_after_clean),
        "class_balance": df['label'].value_counts().to_dict(),
    },
    "model_results": results,
    "top_positive_words": [w for w, c in top_pos],
    "top_negative_words": [w for w, c in top_neg],
    "top_positive_predictors": [(w, round(float(c), 3)) for w, c in top_pos_features],
    "top_negative_predictors": [(w, round(float(c), 3)) for w, c in top_neg_features],
}

In [31]:
with open(f"{OUT}/results_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
 
# Full classification report text for both models, saved for the report
with open(f"{OUT}/classification_reports.txt", "w") as f:
    for name, model in models.items():
        y_pred = model.predict(X_test_tfidf)
        f.write(f"===== {name} =====\n")
        f.write(classification_report(y_test, y_pred, target_names=['neg', 'pos']))
        f.write("\n\n")
 
print("\nAll done. Outputs saved in ./outputs/")
print(json.dumps(results, indent=2))


All done. Outputs saved in ./outputs/
{
  "Logistic Regression": {
    "cv_f1_mean": 0.875294036874233,
    "cv_f1_std": 0.00545577603501843,
    "test_accuracy": 0.868,
    "test_f1": 0.8648925281473899,
    "test_precision": 0.868448098663926,
    "test_recall": 0.8613659531090724
  },
  "Linear SVM": {
    "cv_f1_mean": 0.8712882234277493,
    "cv_f1_std": 0.004237301390568265,
    "test_accuracy": 0.8595,
    "test_f1": 0.8565594691168964,
    "test_precision": 0.8578732106339468,
    "test_recall": 0.855249745158002
  }
}
